# DS Week 08 — SubjectShift: Label-Efficient Wearable Activity Recognition Under Unseen-Person Shift

**Problem framing:** Evaluate cross-person activity recognition with scarce labels. **Industry:** Digital health / wearable sensing. **Date:** 2026-10-31. **Experience target:** ~3 years 8 months since March 2023; Senior/Staff-adjacent Data Scientist. **Primary artifact:** reproducible scientific notebook.

**Hypotheses.** (H1) subject-disjoint evaluation is more credible than window-random splitting because 50%-overlapping windows share personal motion signatures; (H2) dynamics-aware masked reconstruction pretraining may improve frozen linear-probe macro-F1 under 10% label availability; (H3) uncertainty-aware abstention may reduce the expected cost of automated mistakes, at a manual-review cost. We test, not assume, these hypotheses.

**Research:** Adapt the motion-aware masking and cross-sensor reconstruction *idea* of *Dynamics-Guided Cross-Modality Masked State Space Modeling for Robust Wearable HAR* (IEEE AIC 2026; DOI 10.1109/AIC70896.2026.11717057), using a small Conv1D encoder instead of reproducing the paper's state-space architecture. BenchHAR (Cai et al., May 2026, arXiv:2605.08296) motivates evaluation across unseen distributions. **No paper result is attributed to this implementation.**

**Business interpretation:** Activity recognition could support wellness analytics and rehabilitation research, but this dataset is **not a clinical validation set**. Misclassifications and abstentions have distinct costs; illustrative costs below are not medical recommendations.

## 1. Reproducibility, hardware, and dataset policy

Dataset: UCI Human Activity Recognition Using Smartphones (Reyes-Ortiz et al., DOI 10.24432/C54S4K), CC BY 4.0. Official train/test split uses disjoint participants (21/9); 10,299 2.56-second windows (128 samples, 50 Hz), 9 inertial channels. No dataset is committed. To download, run `python -c "from src.har import download; print(download())"` from repository root. **Synthetic fallback is an execution smoke test, not a scientific performance benchmark.**

Runtime: Ryzen 7 4800-series, 16 GB RAM, GTX 1650 Ti 4 GB VRAM. Float32 arrays, Conv1D width 32, small batches, CPU fallback; `FAST_MODE` shortens SSL epochs. Do not use held-out test labels for tuning. All output metrics are tagged with dataset provenance.

In [ ]:
import os, sys, json, time, random, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
import torch
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import ConfusionMatrixDisplay, f1_score

if not Path('src/har.py').exists() and Path('../src/har.py').exists(): os.chdir('..')
assert Path('src/har.py').exists(), 'Launch notebook from project root or notebooks/ folder'
sys.path.insert(0,str(Path('.').resolve()))
from src.har import SEED, LABELS, load_real, synthetic, split_subjects, engineer, rotate_and_noise
from src.learning import Encoder, pretrain, embed, fit_probe, stratified_budget
from src.metrics import evaluate, cluster_ci, selective, temperature_calibrate
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(4,torch.get_num_threads()))
FAST_MODE=True  # False for full research training
REQUIRE_REAL=False  # True fails unless original UCI HAR data are present
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print({'seed':SEED,'device':DEVICE,'fast_mode':FAST_MODE,'torch':torch.__version__})

## 2. Provenance and target definition

Target is one of six activity labels. The UCI sensor windows were collected with a waist-mounted phone; the supplied 9 channels include **body acceleration, angular velocity, and total acceleration**. There are no patient outcomes, no temporal event prediction, and no clinically validated safety label. `subject` is metadata for splitting and clustered uncertainty only. The original held-out test participants remain untouched until the final evaluation.

In [ ]:
root=Path('data/UCI HAR Dataset')
if root.exists(): X,y,subjects,original_split,DATA_MODE=load_real(root)
else:
    if REQUIRE_REAL: raise FileNotFoundError('Real UCI HAR dataset required; see DATASET.md')
    X,y,subjects,original_split,DATA_MODE=synthetic(n_subjects=24,windows_per_subject=36)
assert X.dtype==np.float32 and X.shape[1:]==(128,9)
assert len(X)==len(y)==len(subjects)==len(original_split)
print('DATA MODE:',DATA_MODE,'shape:',X.shape,'subjects:',len(np.unique(subjects)))
if DATA_MODE.startswith('synthetic'): print('WARNING: all numeric results are SYNTHETIC SMOKE TESTS; not real UCI performance.')

## 3. Data quality, missingness, outliers, EDA

Inspect missing values, finite values, window amplitudes, class prevalence, participant coverage, and distribution of motion intensity. Outlier windows are **not removed based on test-set behavior**: vigorous activities legitimately produce large acceleration/jerk. Potential sensor failures must be distinguished from genuine high-motion windows.

In [ ]:
quality={'missing_cells':int(np.isnan(X).sum()),'nonfinite_cells':int((~np.isfinite(X)).sum()),'duplicate_window_count':int(len(X)-len(np.unique(X.reshape(len(X),-1),axis=0))),'n_subjects':int(len(np.unique(subjects))),'n_windows':int(len(X))}
assert quality['nonfinite_cells']==0, 'Investigate invalid sensor readings before fitting'
counts=pd.crosstab(pd.Series(y,name='activity_id'),pd.Series(original_split,name='official_split'))
display(counts.rename(index=dict(enumerate(LABELS))))
display(pd.Series(quality,name='quality'))
energy=np.sqrt((X[:,:,:3]**2).mean(axis=(1,2)))
fig,ax=plt.subplots(figsize=(7,3));ax.hist(energy,bins=30);ax.set(title='Body-acceleration RMS distribution',xlabel='RMS (sensor units)',ylabel='Windows');plt.tight_layout();Path('artifacts').mkdir(exist_ok=True);fig.savefig('artifacts/synthetic_eda.png',dpi=130);plt.show()
print('RMS quantiles:',np.quantile(energy,[0,.01,.5,.99,1]).round(4))

## 4. Subject-aware experimental design and leakage audit

UCI's official test subjects are held out. Training subjects are further split into training, validation, and calibration groups. **No window overlap across subjects**, including overlapping windows from the same person. Self-supervised pretraining uses only training subjects. Hyperparameters and model choice use validation; temperature scaling uses calibration; test is touched once for final reporting. Subject-cluster bootstrap is used instead of treating overlapping windows as independent.

In [ ]:
parts=split_subjects(subjects,original_split,seed=SEED)
partition_audit=pd.DataFrame([{'partition':name,'windows':len(idx),'subjects':len(np.unique(subjects[idx])),'classes':len(np.unique(y[idx]))} for name,idx in parts.items()]);display(partition_audit)
for a,ia in parts.items():
 for b,ib in parts.items():
  if a!=b: assert not (set(subjects[ia])&set(subjects[ib]))
assert len(np.unique(y[parts['train']]))==6, 'All activity classes must occur in train'
train,val,cal,test=[parts[k] for k in ('train','validation','calibration','test')]

## 5. Domain-aware feature engineering and selection

**Why these features?** Axis RMS and 3D magnitude capture motion intensity; jerk RMS captures rapid transitions; dominant frequency and spectral energy capture gait cadence; cross-axis correlation captures coordinated movement; IQR is resistant to transient spikes. Each feature uses only the current 128-sample window and never uses subject identity, target label, or later windows. Imputation and scaling are learned on training subjects only. Feature selection uses validation-set permutation importance, without modifying the test protocol.

In [ ]:
F=engineer(X)
print('Feature matrix:',F.shape,'missing features:',int(F.isna().sum().sum()))
assert all('subject' not in c and 'target' not in c for c in F.columns)
print('Feature families:',list(F.columns[:10]))
models={
 'majority':DummyClassifier(strategy='most_frequent'),
 'logistic':make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),LogisticRegression(max_iter=350,C=1,random_state=SEED)),
 'extra_trees':make_pipeline(SimpleImputer(strategy='median'),ExtraTreesClassifier(n_estimators=90,min_samples_leaf=2,max_features=.8,n_jobs=4,random_state=SEED)),
 'hist_gbdt':make_pipeline(SimpleImputer(strategy='median'),HistGradientBoostingClassifier(max_iter=90,max_leaf_nodes=15,learning_rate=.08,l2_regularization=1,random_state=SEED)),
}
for name,m in models.items():m.fit(F.iloc[train],y[train])
print('Trained classical models on training subjects only.')

## 6. Label-budget experiment and controlled model selection

Compare **10% and 100% training labels**, selected in a class-stratified manner exclusively from training subjects. The 10% comparison includes logistic regression, ExtraTrees, and a frozen random CNN probe. The research adaptation pretrains a masked reconstruction CNN on *unlabeled training-subject windows only*, then fits the same linear probe using the identical labeled subset. This is not a reproduction of a state-space model. Validation macro-F1 is the primary selection criterion. The full-label models are secondary reference experiments.

In [ ]:
idx10=stratified_budget(y,train,.10,SEED)
print('Label budget:',len(idx10),'/',len(train),'(10% approximate, class stratified)')
low_models={
 'logistic_10pct':make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),LogisticRegression(max_iter=350,C=1,random_state=SEED)),
 'extra_trees_10pct':make_pipeline(SimpleImputer(strategy='median'),ExtraTreesClassifier(n_estimators=90,min_samples_leaf=2,n_jobs=4,random_state=SEED)),
}
for name,m in low_models.items():m.fit(F.iloc[idx10],y[idx10])
all_models={**{name+'_100pct':m for name,m in models.items()},**low_models}
print('Classical label-budget models:',list(all_models))

## 7. Research adaptation: dynamics-aware masked reconstruction

The self-supervised task masks 20% of sensor time points with higher sampling probability around local acceleration changes, and reconstructs withheld values from the remaining multichannel signal. A small Conv1D encoder is frozen for downstream probing. The mask is generated from the *unmasked training input*, but only reconstruction targets are used for loss; the classifier never receives activity labels during pretraining. Unlike the referenced 2026 paper, we do **not** implement a selective state-space backbone or claim its published results.

In [ ]:
epochs=2 if FAST_MODE else 8
start=time.perf_counter()
ssl_model,ssl_losses=pretrain(X[train],seed=SEED,epochs=epochs,device=DEVICE,dynamic=True)
ssl_elapsed=round(time.perf_counter()-start,2)
random_encoder=Encoder()
emb_ssl=embed(ssl_model.encoder,X,device=DEVICE)
emb_random=embed(random_encoder,X,device=DEVICE)
probes={
 'ssl_masked_10pct':fit_probe(emb_ssl,y,idx10,seed=SEED),
 'random_encoder_10pct':fit_probe(emb_random,y,idx10,seed=SEED),
 'ssl_masked_100pct':fit_probe(emb_ssl,y,train,seed=SEED),
}
print('SSL reconstruction losses:',ssl_losses,'seconds:',ssl_elapsed)
fig,ax=plt.subplots(figsize=(6,2.5));ax.plot(np.arange(1,len(ssl_losses)+1),ssl_losses,marker='o');ax.set(xlabel='Epoch',ylabel='Masked MSE',title='SSL training (train subjects only)');plt.tight_layout();fig.savefig('artifacts/synthetic_ssl_loss.png',dpi=130);plt.show()

## 8. Validation experiment comparison and constrained hyperparameter strategy

Hyperparameters are deliberately bounded: three classical models, one fixed small encoder, one 10% label budget, and 2/8 pretraining epochs depending on smoke/full mode. This is a pre-specified CPU-friendly benchmark rather than a broad hyperparameter search. The validation subjects determine the selected 10%-label model. The test subjects remain unexamined until after selection and calibration.

In [ ]:
def model_probs(name,idx):
 if name in all_models:return all_models[name].predict_proba(F.iloc[idx])
 if name.startswith('random_encoder'):return probes[name].predict_proba(emb_random[idx])
 return probes[name].predict_proba(emb_ssl[idx])

candidate_names=['logistic_10pct','extra_trees_10pct','random_encoder_10pct','ssl_masked_10pct']
validation=[]
for name in candidate_names:
 m=evaluate(y[val],model_probs(name,val))
 validation.append({'model':name,**m})
validation_df=pd.DataFrame(validation).sort_values('macro_f1',ascending=False)
display(validation_df)
champion=validation_df.iloc[0]['model'];print('10% label-budget validation champion:',champion)
reference=[]
for name in ['majority_100pct','logistic_100pct','extra_trees_100pct','hist_gbdt_100pct','ssl_masked_100pct']:
 reference.append({'model':name,**evaluate(y[val],model_probs(name,val))})
display(pd.DataFrame(reference).sort_values('macro_f1',ascending=False))

## 9. Independent calibration and final unseen-subject test

Use the calibration-subject partition to optimize a **single scalar temperature** on log-probabilities. This cannot alter the predicted class, but may improve confidence reliability. The final held-out test cohort is used only now. We report macro-F1, balanced accuracy, accuracy, NLL, ECE, and a subject-clustered 95% bootstrap CI. Bootstrap resamples subjects, not windows.

In [ ]:
p_cal=model_probs(champion,cal)
p_test_raw=model_probs(champion,test)
temperature,p_test=temperature_calibrate(p_cal,y[cal],p_test_raw)
primary=evaluate(y[test],p_test)
primary['subject_bootstrap_macro_f1_95ci']=cluster_ci(y[test],p_test,subjects[test],seed=SEED,n_boot=120 if FAST_MODE else 400)
primary['temperature']=temperature
primary['model']=champion
primary['dataset_mode']=DATA_MODE
print('FINAL TEST (',DATA_MODE,'):',json.dumps(primary,indent=2))
fig,ax=plt.subplots(figsize=(6,5));ConfusionMatrixDisplay.from_predictions(y[test],p_test.argmax(axis=1),display_labels=LABELS,xticks_rotation=45,ax=ax,cmap='Blues',colorbar=False);plt.tight_layout();fig.savefig('artifacts/synthetic_confusion.png',dpi=130);plt.show()

## 10. Calibration and selective prediction / decision cost

Expected cost is illustrated as `10 × incorrect auto-decision + 1 × human-reviewed window`. This does **not** represent clinical or industrial cost estimates. A prediction can be routed for manual review if maximum calibrated probability is below a threshold. Compare coverage and cost across pre-specified thresholds; **do not optimize thresholds using the final test set**. For real deployment, thresholds must be chosen using calibration data and validated prospectively.

In [ ]:
thresholds=[.6,.7,.8,.9]
_,p_cal_calibrated=temperature_calibrate(p_cal,y[cal],p_cal)
cal_costs=[{'threshold':t,**selective(y[cal],p_cal_calibrated,t)} for t in thresholds]
cal_cost_df=pd.DataFrame(cal_costs);display(cal_cost_df)
selected_threshold=float(cal_cost_df.sort_values('cost_per_window').iloc[0]['threshold'])
selective_test=selective(y[test],p_test,selected_threshold)
print('Preselected threshold from calibration:',selected_threshold,'test:',selective_test)
conf=p_test.max(axis=1);correct=(p_test.argmax(axis=1)==y[test]);edges=np.linspace(0,1,9)
xs=[];ys=[];ns=[]
for lo,hi in zip(edges[:-1],edges[1:]):
 mask=(conf>=lo)&(conf<(hi if hi<1 else 1.01))
 if mask.any():xs.append(float(conf[mask].mean()));ys.append(float(correct[mask].mean()));ns.append(int(mask.sum()))
fig,ax=plt.subplots(figsize=(4,4));ax.plot([0,1],[0,1],'--',label='ideal');ax.plot(xs,ys,'o-',label='observed');ax.set(xlabel='Mean confidence',ylabel='Accuracy',title='Test reliability (descriptive)');ax.legend();plt.tight_layout();fig.savefig('artifacts/synthetic_reliability.png',dpi=130);plt.show()

## 11. Interpretability and feature selection

Permutation importance on validation subjects identifies engineered inertial features associated with macro-F1. This is an association, not causal evidence. We use it to understand potential failure modes; we do not refit on the test set. Neural representation probing is less transparent, so a classical feature model is an important interpretability reference.

In [ ]:
reference_model=models['extra_trees']
imp=permutation_importance(reference_model,F.iloc[val],y[val],scoring='f1_macro',n_repeats=2 if FAST_MODE else 5,random_state=SEED,n_jobs=2)
importance=pd.Series(imp.importances_mean,index=F.columns).sort_values(ascending=False)
display(importance.head(12).to_frame('validation permutation Δmacro-F1'))
fig,ax=plt.subplots(figsize=(7,4));importance.head(12).iloc[::-1].plot.barh(ax=ax);ax.set(xlabel='Validation permutation importance',title='Inertial feature importance');plt.tight_layout();fig.savefig('artifacts/synthetic_importance.png',dpi=130);plt.show()

## 12. Robustness and slice/error analysis

Pre-register perturbations: shared 3D rotation for all sensors (mimics changed orientation), low Gaussian noise, and complete gyroscope dropout. They are illustrative stress tests, not validated hardware failure distributions. Slice analysis groups by held-out subject and by activity, reporting where the model is unreliable. No robustness variant is used to retrain or select the model.

In [ ]:
def score_signal(z):
 if champion in all_models:return all_models[champion].predict_proba(engineer(z))
 if champion.startswith('random_encoder'):return probes[champion].predict_proba(embed(random_encoder,z,device=DEVICE))
 return probes[champion].predict_proba(embed(ssl_model.encoder,z,device=DEVICE))
robustness=[]
for name,kwargs in [('clean',dict(noise=0,rotation=False)),('rotation',dict(noise=0,rotation=True)),('noise',dict(noise=.08,rotation=False)),('gyro_dropout',dict(noise=0,rotation=False,drop_gyro=True))]:
 z=X[test] if name=='clean' else rotate_and_noise(X[test],seed=SEED,**kwargs)
 pp=score_signal(z)
 robustness.append({'condition':name,**evaluate(y[test],pp)})
robust_df=pd.DataFrame(robustness);display(robust_df)
slices=[]
for subject in np.unique(subjects[test]):
 mask=subjects[test]==subject
 slices.append({'subject':int(subject),'n':int(mask.sum()),'macro_f1':float(f1_score(y[test][mask],p_test[mask].argmax(axis=1),average='macro',zero_division=0))})
display(pd.DataFrame(slices).sort_values('macro_f1').head(8))
class_errors=pd.DataFrame({'activity':[LABELS[i] for i in y[test]],'error':p_test.argmax(axis=1)!=y[test]}).groupby('activity').agg(n=('error','size'),error_rate=('error','mean'))
display(class_errors.sort_values('error_rate',ascending=False))

## 13. Conclusions, limitations, and production/research follow-ups

**Interpret only the actual numbers above.** The selected model may be classical or SSL; superiority is not assumed. Real-data results are valid only when `DATA_MODE == 'uci_har_real'`. Synthetic results verify the pipeline, not any hypothesis.

**Limitations:** 30 volunteers; old waist-mounted phone; narrow age/device variation; overlapping windows and session effects; no site/device diversity; accelerometer axes may change under real-world wear; single fixed test cohort and pretraining seed; no external clinical outcome; calibration small due to subject-group constraints.

**Next experiments:** leave-one-subject-out repeated outer evaluation for variance; multiple SSL seeds and label fractions; subject-invariant contrastive objective; cross-device transfer to HARTH or HHAR; compare with 2026 masked state-space models; prospective device-rotation and missing-sensor evaluation; user-specific calibration only with consent and without contaminating benchmark test labels. A real clinical or workplace deployment needs prospective safety and fairness assessment.

In [ ]:
out=Path('artifacts');out.mkdir(exist_ok=True)
summary={'dataset_mode':DATA_MODE,'seed':SEED,'hardware_device':DEVICE,'fast_mode':FAST_MODE,'quality':quality,'split_audit':partition_audit.to_dict(orient='records'),'validation_10pct':validation_df.to_dict(orient='records'),'validation_100pct':reference,'test_primary':primary,'calibration_costs':cal_costs,'test_selective':selective_test,'robustness':robustness,'subject_slices':slices,'ssl_losses':ssl_losses,'ssl_seconds':ssl_elapsed,'limitations':'Synthetic metrics are smoke only' if DATA_MODE.startswith('synthetic') else 'Subject-limited single-benchmark evaluation'}
name='synthetic_smoke_metrics.json' if DATA_MODE.startswith('synthetic') else 'uci_har_real_metrics.json'
(out/name).write_text(json.dumps(summary,indent=2,default=float))
validation_df.to_csv(out/('synthetic_validation.csv' if DATA_MODE.startswith('synthetic') else 'uci_har_validation.csv'),index=False)
print('Saved:',out/name,'; results must be labeled',DATA_MODE)
